# Object-Oriented Programming (OOP)

Object-Oriented Programming organises code around **objects** — bundles of data (attributes) and behaviour (methods). Python is a fully object-oriented language — everything, including integers and functions, is an object. This notebook covers OOP from first principles all the way to advanced patterns.

## What You'll Learn
1. Classes & instances — the blueprint and the thing
2. `__init__` and instance attributes
3. Methods — instance, class, and static
4. Properties — controlled attribute access
5. Dunder (magic) methods
6. Inheritance & `super()`
7. Multiple inheritance & MRO
8. Abstract base classes
9. Mixins
10. Composition vs inheritance
11. `dataclasses`
12. `__slots__`
13. Class decorators — `@classmethod`, `@staticmethod`, `@property`
14. Metaclasses
15. Protocols (structural subtyping)
16. Design patterns in Python

---

## 1. Classes & Instances

A **class** is a blueprint. An **instance** is an object built from that blueprint. Every instance gets its own copy of the instance attributes, but shares the class's methods.

In [ ]:
# Minimal class definition
class Dog:
    pass

# Create instances
dog1 = Dog()
dog2 = Dog()

print(type(dog1))           # <class '__main__.Dog'>
print(isinstance(dog1, Dog)) # True
print(dog1 is dog2)          # False — distinct objects
print(id(dog1), id(dog2))    # different memory addresses

In [ ]:
# Everything in Python is an object
for obj in [42, 3.14, "hello", [1, 2], {"a": 1}, Dog, Dog()]:
    print(f"{str(obj):<20} type={type(obj).__name__}")

---

## 2. `__init__` and Instance Attributes

`__init__` is called automatically when an instance is created. It initialises the instance's attributes. `self` is the instance itself — always the first parameter of instance methods.

In [ ]:
class Dog:
    # Class attribute — shared by ALL instances
    species = "Canis lupus familiaris"
    count   = 0

    def __init__(self, name: str, breed: str, age: int):
        # Instance attributes — unique to each instance
        self.name  = name
        self.breed = breed
        self.age   = age
        Dog.count += 1          # update class attribute via class name


rex   = Dog("Rex",   "German Shepherd", 3)
bella = Dog("Bella", "Golden Retriever", 5)
max_  = Dog("Max",   "Labrador",         2)

print(f"name:    {rex.name}")
print(f"breed:   {rex.breed}")
print(f"age:     {rex.age}")
print(f"species: {rex.species}")    # from class attribute
print(f"count:   {Dog.count}")      # 3 — shared across all instances

In [ ]:
# Class vs instance attribute lookup order
# Python looks in the instance __dict__ first, then the class __dict__

print("\nInstance __dict__:")
print(rex.__dict__)    # only instance attributes

print("\nClass __dict__ (abridged):")
class_items = {k: v for k, v in Dog.__dict__.items()
               if not k.startswith('__')}
print(class_items)

# Instance attribute shadows class attribute
rex.species = "Domestic Dog"   # sets on instance, not class
print(f"\nrex.species:   {rex.species}")    # 'Domestic Dog'
print(f"bella.species: {bella.species}")   # still 'Canis lupus familiaris'
print(f"Dog.species:   {Dog.species}")     # class unchanged

---

## 3. Methods — Instance, Class, and Static

In [ ]:
import math
from datetime import date


class Circle:
    """A circle with radius, area, and circumference."""

    PI = math.pi  # class attribute

    def __init__(self, radius: float):
        if radius <= 0:
            raise ValueError(f"radius must be positive, got {radius}")
        self.radius = radius

    # ── Instance method ───────────────────────────────────────────────────
    # Receives 'self' — operates on the specific instance
    def area(self) -> float:
        """Return the area of the circle."""
        return self.PI * self.radius ** 2

    def circumference(self) -> float:
        """Return the circumference of the circle."""
        return 2 * self.PI * self.radius

    def scale(self, factor: float) -> "Circle":
        """Return a new Circle scaled by factor."""
        return Circle(self.radius * factor)

    # ── Class method ──────────────────────────────────────────────────────
    # Receives 'cls' (the class itself) — used for alternative constructors
    @classmethod
    def from_diameter(cls, diameter: float) -> "Circle":
        """Create a Circle from its diameter instead of radius."""
        return cls(diameter / 2)

    @classmethod
    def unit(cls) -> "Circle":
        """Return a unit circle (radius = 1)."""
        return cls(1.0)

    # ── Static method ─────────────────────────────────────────────────────
    # Receives neither self nor cls — logically belongs to the class
    # but doesn't need access to instance or class state
    @staticmethod
    def is_valid_radius(r: float) -> bool:
        """Return True if r is a valid (positive) radius."""
        return isinstance(r, (int, float)) and r > 0


c = Circle(5)
print(f"area:          {c.area():.4f}")
print(f"circumference: {c.circumference():.4f}")
print(f"scaled x2:     radius={c.scale(2).radius}")

c2 = Circle.from_diameter(10)   # classmethod — alternative constructor
print(f"from_diameter: radius={c2.radius}")

print(f"unit circle:   area={Circle.unit().area():.4f}")
print(f"valid radius:  {Circle.is_valid_radius(5)}, {Circle.is_valid_radius(-1)}")

In [ ]:
# Calling methods — two equivalent ways
c = Circle(3)

# ✅ Normal call — Python automatically passes self
print(c.area())

# Same thing written explicitly (unbound method call)
print(Circle.area(c))

# Methods are just functions stored on the class
print(type(Circle.area))     # <class 'function'>
print(type(c.area))          # <class 'method'> — bound to c

---

## 4. Properties — Controlled Attribute Access

`@property` turns a method into an attribute-like accessor. It lets you add validation, computation, or lazy loading behind a clean `obj.attr` interface — without breaking existing code.

In [ ]:
class Temperature:
    """Temperature that stores Celsius internally but exposes Fahrenheit."""

    def __init__(self, celsius: float):
        self.celsius = celsius   # goes through the setter below

    @property
    def celsius(self) -> float:
        """Temperature in Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError(f"Temperature below absolute zero: {value}")
        self._celsius = value

    @celsius.deleter
    def celsius(self) -> None:
        raise AttributeError("Cannot delete temperature")

    @property
    def fahrenheit(self) -> float:
        """Temperature in Fahrenheit (computed from Celsius)."""
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value: float) -> None:
        self.celsius = (value - 32) * 5 / 9   # convert and validate via celsius setter

    @property
    def kelvin(self) -> float:
        """Temperature in Kelvin (read-only)."""
        return self._celsius + 273.15


t = Temperature(100)
print(f"Celsius:    {t.celsius}")
print(f"Fahrenheit: {t.fahrenheit}")
print(f"Kelvin:     {t.kelvin}")

t.fahrenheit = 32        # set via Fahrenheit
print(f"\nSet 32°F → {t.celsius}°C")

try:
    t.celsius = -300     # triggers validation
except ValueError as e:
    print(f"ValueError: {e}")

try:
    t.kelvin = 100       # read-only — no setter defined
except AttributeError as e:
    print(f"AttributeError: {e}")

In [ ]:
# Lazy property — compute expensive value only once, then cache it
class DataSet:
    """A dataset that computes statistics lazily."""

    def __init__(self, data: list[float]):
        self._data = data
        self._stats = None   # not yet computed

    @property
    def stats(self) -> dict:
        """Compute and cache descriptive statistics."""
        if self._stats is None:          # compute on first access
            print("  (computing stats...)")
            n = len(self._data)
            mean = sum(self._data) / n
            variance = sum((x - mean) ** 2 for x in self._data) / n
            self._stats = {
                "n":      n,
                "mean":   round(mean, 4),
                "std":    round(variance ** 0.5, 4),
                "min":    min(self._data),
                "max":    max(self._data),
            }
        return self._stats


ds = DataSet([4, 8, 15, 16, 23, 42])
print("First access:")
print(ds.stats)
print("Second access (cached):")
print(ds.stats)

---

## 5. Dunder (Magic) Methods

Dunder methods (double underscore on both sides) let your objects integrate with Python's built-in syntax and functions — `len()`, `+`, `[]`, `with`, iteration, and more.

In [ ]:
class Vector:
    """A 2-D mathematical vector."""

    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    # ── String representations ────────────────────────────────────────────
    def __repr__(self) -> str:
        """Unambiguous string — for developers (and eval())."""
        return f"Vector({self.x!r}, {self.y!r})"

    def __str__(self) -> str:
        """Readable string — for end users."""
        return f"({self.x}, {self.y})"

    # ── Arithmetic operators ──────────────────────────────────────────────
    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector") -> "Vector":
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar: float) -> "Vector":
        """Vector * scalar."""
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar: float) -> "Vector":
        """scalar * Vector — called when scalar doesn't know how to multiply."""
        return self.__mul__(scalar)

    def __neg__(self) -> "Vector":
        return Vector(-self.x, -self.y)

    def __abs__(self) -> float:
        """Magnitude (length) of the vector."""
        return (self.x ** 2 + self.y ** 2) ** 0.5

    # ── Comparison ────────────────────────────────────────────────────────
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Vector):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __lt__(self, other: "Vector") -> bool:
        """Compare by magnitude."""
        return abs(self) < abs(other)

    # ── Container protocol ────────────────────────────────────────────────
    def __len__(self) -> int:
        """A 2-D vector always has 2 components."""
        return 2

    def __getitem__(self, index: int) -> float:
        return (self.x, self.y)[index]

    def __iter__(self):
        yield self.x
        yield self.y

    # ── Boolean ───────────────────────────────────────────────────────────
    def __bool__(self) -> bool:
        """A zero vector is falsy."""
        return self.x != 0 or self.y != 0

    # ── Hashing (needed to use in sets/dicts) ─────────────────────────────
    def __hash__(self) -> int:
        return hash((self.x, self.y))


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"repr:      {v1!r}")
print(f"str:       {v1}")
print(f"add:       {v1 + v2}")
print(f"sub:       {v1 - v2}")
print(f"mul:       {v1 * 3}")
print(f"rmul:      {3 * v1}")
print(f"neg:       {-v1}")
print(f"abs:       {abs(v1)}")
print(f"eq:        {v1 == Vector(3, 4)}")
print(f"lt:        {v2 < v1}")
print(f"len:       {len(v1)}")
print(f"index [0]: {v1[0]}")
print(f"iter:      {list(v1)}")
print(f"unpack:    ", end=""); x, y = v1; print(f"x={x}, y={y}")
print(f"bool:      {bool(v1)}, {bool(Vector(0, 0))}")
print(f"in set:    {v1 in {v1, v2}}")

In [ ]:
# ── Context manager dunder methods ────────────────────────────────────────

class ManagedFile:
    """A file wrapper that tracks access and auto-closes."""

    def __init__(self, path: str, mode: str = 'r'):
        self.path  = path
        self.mode  = mode
        self._file = None

    def __enter__(self):
        """Open the file when entering the with block."""
        print(f"  Opening {self.path}")
        self._file = open(self.path, self.mode, encoding='utf-8')
        return self._file   # value bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close the file when leaving the with block."""
        print(f"  Closing {self.path}")
        if self._file:
            self._file.close()
        # return True to suppress the exception; False/None to propagate it
        return False


import tempfile, os
tmp = tempfile.mktemp(suffix='.txt')

with ManagedFile(tmp, 'w') as f:
    f.write('hello from ManagedFile\n')

with ManagedFile(tmp) as f:
    print(f.read())

os.unlink(tmp)

In [ ]:
# ── Dunder method reference ───────────────────────────────────────────────
dunder_ref = """
LIFECYCLE
  __new__(cls, ...)     → creates the instance (before __init__)
  __init__(self, ...)   → initialises the instance
  __del__(self)         → called when instance is garbage-collected

REPRESENTATION
  __repr__(self)        → repr(obj), developer-facing
  __str__(self)         → str(obj), user-facing
  __format__(self, fmt) → format(obj, fmt), f-strings
  __bytes__(self)       → bytes(obj)

COMPARISONS
  __eq__(self, other)   → obj == other
  __ne__(self, other)   → obj != other
  __lt__, __le__        → <, <=
  __gt__, __ge__        → >, >=
  __hash__(self)        → hash(obj) — needed for set/dict membership

ARITHMETIC
  __add__, __radd__     → obj + other, other + obj
  __sub__, __rsub__     → obj - other, other - obj
  __mul__, __rmul__     → obj * other, other * obj
  __truediv__           → obj / other
  __floordiv__          → obj // other
  __mod__               → obj % other
  __pow__               → obj ** other
  __neg__, __pos__      → -obj, +obj
  __abs__               → abs(obj)
  __iadd__, __isub__ ...
                        → obj += other (in-place variants)

CONTAINERS
  __len__(self)         → len(obj)
  __getitem__(self, k)  → obj[k]
  __setitem__(self, k, v) → obj[k] = v
  __delitem__(self, k)  → del obj[k]
  __contains__(self, x) → x in obj
  __iter__(self)        → iter(obj), for loops
  __next__(self)        → next(obj)
  __reversed__(self)    → reversed(obj)

ATTRIBUTE ACCESS
  __getattr__(self, name)     → obj.name (only when not found normally)
  __setattr__(self, name, v)  → obj.name = v
  __delattr__(self, name)     → del obj.name
  __getattribute__(self, name)→ every attribute access

CALLABLE
  __call__(self, ...)   → obj(...) — makes instance callable

CONTEXT MANAGER
  __enter__(self)       → with obj as x:
  __exit__(self, ...)   → leaving the with block

BOOLEAN / NUMERIC
  __bool__(self)        → bool(obj), if obj:
  __int__, __float__    → int(obj), float(obj)

COPY
  __copy__(self)        → copy.copy(obj)
  __deepcopy__(self, m) → copy.deepcopy(obj)
"""
print(dunder_ref)

---

## 6. Inheritance & `super()`

Inheritance lets a class reuse and extend another class's behaviour. The child class **is-a** specialisation of the parent.

In [ ]:
class Animal:
    """Base class for all animals."""

    def __init__(self, name: str, sound: str):
        self.name  = name
        self.sound = sound

    def speak(self) -> str:
        return f"{self.name} says {self.sound}!"

    def describe(self) -> str:
        return f"I am {self.name}, a {type(self).__name__}."

    def __repr__(self) -> str:
        return f"{type(self).__name__}(name={self.name!r})"


class Dog(Animal):
    """A dog — extends Animal with fetch behaviour."""

    def __init__(self, name: str, breed: str):
        super().__init__(name, sound="Woof")  # call parent __init__
        self.breed = breed

    def fetch(self, item: str) -> str:
        return f"{self.name} fetches the {item}!"

    def describe(self) -> str:
        base = super().describe()             # call parent method
        return f"{base} Breed: {self.breed}."


class Cat(Animal):
    """A cat — overrides speak to be more feline."""

    def __init__(self, name: str, indoor: bool = True):
        super().__init__(name, sound="Meow")
        self.indoor = indoor

    def speak(self) -> str:
        base = super().speak()                # start from parent version
        mood = "softly" if self.indoor else "loudly"
        return f"{base} ({mood})"


animals = [
    Dog("Rex",   "German Shepherd"),
    Dog("Bella", "Poodle"),
    Cat("Whiskers"),
    Cat("Stray", indoor=False),
    Animal("Bird", "Tweet"),
]

for animal in animals:
    print(animal.speak())

print()
print(animals[0].describe())
print(animals[0].fetch("ball"))
print()

# isinstance checks hierarchy
rex = animals[0]
print(f"isinstance(rex, Dog):    {isinstance(rex, Dog)}")
print(f"isinstance(rex, Animal): {isinstance(rex, Animal)}")
print(f"isinstance(rex, Cat):    {isinstance(rex, Cat)}")
print(f"issubclass(Dog, Animal): {issubclass(Dog, Animal)}")

In [ ]:
# Polymorphism — same interface, different behaviour
# Works because each class implements speak() its own way

def make_noise(animals: list) -> None:
    """Tell each animal to speak — don't care about the concrete type."""
    for animal in animals:
        print(f"  {animal.speak()}")

print("All animals speaking:")
make_noise(animals)

---

## 7. Multiple Inheritance & MRO

Python supports multiple inheritance. The **Method Resolution Order (MRO)** determines which class's method is called when multiple parents define the same name. Python uses the **C3 linearisation** algorithm.

In [ ]:
class Flyable:
    def move(self) -> str:
        return "flying"

    def describe(self) -> str:
        return "I can fly."


class Swimmable:
    def move(self) -> str:
        return "swimming"

    def describe(self) -> str:
        return "I can swim."


class Duck(Flyable, Swimmable):   # inherits from BOTH
    def speak(self) -> str:
        return "Quack!"


donald = Duck()
print(donald.speak())
print(donald.move())        # Flyable wins — it's listed first
print(donald.describe())    # Flyable wins

# The MRO shows the search order
print("\nMRO:")
for cls in Duck.__mro__:
    print(f"  {cls}")

In [ ]:
# super() with multiple inheritance — cooperative calling
# Each class in the MRO calls super().__init__() to pass control along

class Base:
    def __init__(self):
        print("  Base.__init__")
        super().__init__()   # cooperative — calls next in MRO (object)

class Left(Base):
    def __init__(self):
        print("  Left.__init__")
        super().__init__()   # cooperative — calls next: Right

class Right(Base):
    def __init__(self):
        print("  Right.__init__")
        super().__init__()   # cooperative — calls next: Base

class Child(Left, Right):
    def __init__(self):
        print("  Child.__init__")
        super().__init__()   # starts the cooperative chain

print("MRO:", [c.__name__ for c in Child.__mro__])
print("\nInstantiating Child:")
c = Child()
# Each class is called exactly ONCE because of cooperative super() calls

---

## 8. Abstract Base Classes

An **abstract class** defines an interface that subclasses must implement. You cannot instantiate an abstract class — it exists only to be subclassed.

In [ ]:
from abc import ABC, abstractmethod


class Shape(ABC):
    """Abstract base class for all 2-D shapes."""

    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""
        ...

    @abstractmethod
    def perimeter(self) -> float:
        """Return the perimeter of the shape."""
        ...

    # Concrete method — shared by all subclasses
    def describe(self) -> str:
        return (
            f"{type(self).__name__}: "
            f"area={self.area():.2f}, "
            f"perimeter={self.perimeter():.2f}"
        )

    # Abstract class method
    @classmethod
    @abstractmethod
    def from_string(cls, s: str) -> "Shape":
        """Parse a shape from a string representation."""
        ...


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width  = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

    @classmethod
    def from_string(cls, s: str) -> "Rectangle":
        w, h = map(float, s.split('x'))
        return cls(w, h)


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

    @classmethod
    def from_string(cls, s: str) -> "Circle":
        return cls(float(s))


# Cannot instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print(f"TypeError: {e}")

shapes: list[Shape] = [
    Rectangle(4, 5),
    Rectangle.from_string("3x7"),
    Circle(3),
    Circle.from_string("5"),
]

for shape in shapes:
    print(shape.describe())

---

## 9. Mixins

A **mixin** is a class that provides methods to other classes without being a parent in the primary hierarchy. Mixins are small, focused, and not intended to stand alone.

In [ ]:
import json
import copy


class JsonMixin:
    """Adds JSON serialisation to any class with a __dict__."""

    def to_json(self, indent: int = 2) -> str:
        return json.dumps(self.__dict__, indent=indent, default=str)

    @classmethod
    def from_json(cls, json_str: str) -> "JsonMixin":
        data = json.loads(json_str)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj


class ReprMixin:
    """Adds a nice __repr__ based on __init__ parameters."""

    def __repr__(self) -> str:
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{type(self).__name__}({attrs})"


class ComparableMixin:
    """Adds full comparison operators from __eq__ and __lt__."""

    def __le__(self, other) -> bool:
        return self == other or self < other

    def __gt__(self, other) -> bool:
        return not self <= other

    def __ge__(self, other) -> bool:
        return not self < other


class CopyMixin:
    """Adds clone() convenience method."""

    def clone(self):
        return copy.deepcopy(self)


# Combine mixins into a concrete class
class Product(JsonMixin, ReprMixin, ComparableMixin, CopyMixin):
    """A product with price comparison and JSON serialisation."""

    def __init__(self, name: str, price: float, category: str):
        self.name     = name
        self.price    = price
        self.category = category

    def __eq__(self, other) -> bool:
        return isinstance(other, Product) and self.price == other.price

    def __lt__(self, other: "Product") -> bool:
        return self.price < other.price


apple  = Product("Apple",  0.99, "fruit")
laptop = Product("Laptop", 999.0, "electronics")
phone  = Product("Phone",  699.0, "electronics")

print(repr(apple))
print()
print(apple.to_json())
print()
print(f"apple < laptop: {apple < laptop}")
print(f"laptop > phone: {laptop > phone}")

clone = laptop.clone()
clone.price = 899.0
print(f"\nOriginal: {laptop.price}")
print(f"Clone:    {clone.price}")

---

## 10. Composition vs Inheritance

**Inheritance** models an *is-a* relationship. **Composition** models a *has-a* relationship. Composition is often more flexible and easier to test.

In [ ]:
# ── Inheritance (is-a) ────────────────────────────────────────────────────
# Use when the child truly IS a specialisation of the parent.

class Vehicle:
    def __init__(self, make: str, model: str, year: int):
        self.make  = make
        self.model = model
        self.year  = year

class Car(Vehicle):           # A Car IS a Vehicle ✅
    def __init__(self, make, model, year, doors):
        super().__init__(make, model, year)
        self.doors = doors


# ── Composition (has-a) ───────────────────────────────────────────────────
# Use when one object USES another object.

class Engine:
    """An engine component — its own testable unit."""

    def __init__(self, horsepower: int, fuel: str):
        self.horsepower = horsepower
        self.fuel       = fuel
        self._running   = False

    def start(self) -> str:
        self._running = True
        return f"{self.fuel} engine ({self.hp}hp) started"

    def stop(self) -> str:
        self._running = False
        return "Engine stopped"

    @property
    def hp(self):
        return self.horsepower


class GPS:
    """Navigation component."""

    def __init__(self):
        self._destination = None

    def navigate_to(self, destination: str) -> str:
        self._destination = destination
        return f"Navigating to {destination}"


class ElectricCar:
    """An electric car built via composition."""

    def __init__(self, make: str, model: str, range_km: int):
        self.make     = make
        self.model    = model
        self.range_km = range_km
        # COMPOSITION: car HAS an engine and HAS a GPS
        self.engine   = Engine(horsepower=300, fuel="Electric")
        self.gps      = GPS()

    def start(self) -> str:
        return self.engine.start()   # delegate to component

    def navigate(self, destination: str) -> str:
        return self.gps.navigate_to(destination)

    def __repr__(self) -> str:
        return f"ElectricCar({self.make} {self.model}, range={self.range_km}km)"


tesla = ElectricCar("Tesla", "Model S", 650)
print(tesla)
print(tesla.start())
print(tesla.navigate("San Francisco"))
print(f"Engine HP: {tesla.engine.hp}")

In [ ]:
# When to use which:
guidance = """
USE INHERITANCE when:
  ✅ Child truly IS a more specific version of parent
  ✅ You want to extend and override parent behaviour
  ✅ Liskov Substitution Principle holds: child can replace parent
  ✅ Small, focused class hierarchies (max 2–3 levels deep)

USE COMPOSITION when:
  ✅ One object USES or HAS another object
  ✅ You want to swap implementations easily
  ✅ The relationship doesn't pass the 'is-a' test
  ✅ You want to avoid deep inheritance chains
  ✅ Each component can be tested independently

RED FLAGS for inheritance:
  ❌ Child class needs to suppress/ignore parent methods
  ❌ Inheritance chain deeper than 3 levels
  ❌ 'is-a' relationship doesn't hold over time
  ❌ You're inheriting just to reuse code (use composition instead)
"""
print(guidance)

---

## 11. `dataclasses`

`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` and more from annotated fields. Perfect for data-holding classes.

In [ ]:
from dataclasses import dataclass, field, asdict, astuple, replace
from typing import ClassVar


@dataclass
class Point:
    """A 2-D point — minimal dataclass."""
    x: float
    y: float


p1 = Point(3.0, 4.0)
p2 = Point(3.0, 4.0)
p3 = Point(1.0, 2.0)

print(p1)              # __repr__ auto-generated
print(p1 == p2)        # __eq__ auto-generated — True
print(p1 == p3)        # False
print(p1.x, p1.y)

In [ ]:
@dataclass(order=True, frozen=True)   # frozen → immutable + hashable
class Version:
    """
    Semantic version number.

    frozen=True makes it immutable and hashable (usable in sets/dicts).
    order=True generates __lt__, __le__, __gt__, __ge__ automatically.
    """
    major: int
    minor: int
    patch: int

    def __str__(self) -> str:
        return f"{self.major}.{self.minor}.{self.patch}"


v1 = Version(1, 0, 0)
v2 = Version(2, 3, 1)
v3 = Version(1, 0, 0)

print(f"{v1} < {v2}: {v1 < v2}")
print(f"{v1} == {v3}: {v1 == v3}")
print(f"sorted: {sorted([v2, v1, Version(1,5,0)])}")

# frozen=True prevents modification
try:
    v1.major = 99
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# Use replace() to make a modified copy
v1_patched = replace(v1, patch=1)
print(f"Patched: {v1_patched}")

In [ ]:
from datetime import datetime


@dataclass
class Employee:
    """Full-featured dataclass example."""

    # ClassVar — shared across instances, not included in __init__
    company: ClassVar[str] = "Acme Corp"
    _id_counter: ClassVar[int] = 0

    # Required fields (no default)
    name: str
    department: str

    # Optional fields (with defaults)
    salary: float = 50_000.0
    active: bool  = True

    # field() for mutable defaults or extra control
    skills: list[str] = field(default_factory=list)
    notes: str        = field(default="", repr=False)   # excluded from repr

    # Field computed in __post_init__ (not a constructor param)
    employee_id: int    = field(init=False)
    joined_at: datetime = field(init=False)

    def __post_init__(self):
        """Called after __init__ — for derived fields and validation."""
        Employee._id_counter += 1
        self.employee_id = Employee._id_counter
        self.joined_at   = datetime.now()
        if self.salary < 0:
            raise ValueError(f"salary must be non-negative, got {self.salary}")

    def give_raise(self, pct: float) -> None:
        self.salary *= (1 + pct / 100)


alice = Employee("Alice", "Engineering", salary=95_000, skills=["Python", "SQL"])
bob   = Employee("Bob",   "Marketing",   salary=72_000)

print(alice)
print(f"ID: {alice.employee_id}")
print(f"Joined: {alice.joined_at:%Y-%m-%d}")
print(f"Company: {Employee.company}")

alice.give_raise(10)
print(f"After 10% raise: ${alice.salary:,.0f}")

# Serialise to dict / tuple
print(f"\nasdict: {asdict(bob)}")
print(f"astuple: {astuple(bob)[:4]}...")

---

## 12. `__slots__`

`__slots__` restricts instance attributes to a fixed set, reducing memory usage and speeding up attribute access — useful for classes with millions of instances.

In [ ]:
import sys


# Without __slots__ — uses a dict per instance
class PointDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y


# With __slots__ — stores attributes in a fixed array
class PointSlots:
    __slots__ = ('x', 'y')   # only these attributes are allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y


pd = PointDict(1, 2)
ps = PointSlots(1, 2)

print(f"PointDict  size: {sys.getsizeof(pd)} bytes")
print(f"PointSlots size: {sys.getsizeof(ps)} bytes")
print(f"PointDict  has __dict__: {hasattr(pd, '__dict__')}")
print(f"PointSlots has __dict__: {hasattr(ps, '__dict__')}")

# Cannot add arbitrary attributes to slotted class
pd.z = 99    # fine
try:
    ps.z = 99
except AttributeError as e:
    print(f"AttributeError: {e}")

In [ ]:
# Memory comparison at scale
import tracemalloc

N = 100_000

tracemalloc.start()
points_dict = [PointDict(i, i) for i in range(N)]
_, peak_dict = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
points_slots = [PointSlots(i, i) for i in range(N)]
_, peak_slots = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"{N:,} instances:")
print(f"  Without __slots__: {peak_dict / 1e6:.1f} MB")
print(f"  With __slots__:    {peak_slots / 1e6:.1f} MB")
print(f"  Savings: {(1 - peak_slots/peak_dict)*100:.0f}%")

---

## 13. Descriptors

A **descriptor** is an object that customises attribute access on a class. `property` is just a built-in descriptor. Writing your own lets you reuse validation or transformation logic across many attributes and classes.

In [ ]:
# A reusable validation descriptor
class Positive:
    """Descriptor: ensures an attribute is always a positive number."""

    def __set_name__(self, owner, name):
        """Called when the descriptor is assigned to a class attribute."""
        self._name = name
        self._private = f"_{name}"   # store value under _name

    def __get__(self, obj, objtype=None):
        if obj is None:              # accessed on the class itself
            return self
        return getattr(obj, self._private, None)

    def __set__(self, obj, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self._name} must be numeric, got {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"{self._name} must be positive, got {value}")
        setattr(obj, self._private, value)


class Rectangle:
    # Reuse the same descriptor for multiple attributes
    width  = Positive()
    height = Positive()

    def __init__(self, width: float, height: float):
        self.width  = width     # goes through Positive.__set__
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    def __repr__(self):
        return f"Rectangle({self.width}, {self.height})"


r = Rectangle(4, 5)
print(r, "area =", r.area)

r.width = 10
print(r)

for bad_value, exc in [(-1, ValueError), (0, ValueError), ("x", TypeError)]:
    try:
        r.width = bad_value
    except (ValueError, TypeError) as e:
        print(f"{exc.__name__}: {e}")

---

## 14. Metaclasses

A **metaclass** is the class of a class. Just as instances are created by classes, classes themselves are created by metaclasses. The default metaclass is `type`. Metaclasses are advanced — use them only when decorators or class methods can't solve your problem.

In [ ]:
# type() is both a function and the default metaclass
print(type(42))          # <class 'int'>
print(type(int))         # <class 'type'>
print(type(type))        # <class 'type'>  — type is its own metaclass!

# Create a class dynamically with type(name, bases, namespace)
DynamicClass = type(
    'DynamicClass',       # class name
    (object,),            # base classes
    {'greet': lambda self: f"Hello from {type(self).__name__}!"},
)

obj = DynamicClass()
print(obj.greet())

In [ ]:
# A metaclass that enforces naming conventions
class StrictNaming(type):
    """
    Metaclass that enforces:
    - Class name must be PascalCase
    - Public methods must be snake_case
    """

    def __new__(mcs, name, bases, namespace):
        # Check class name
        if not name[0].isupper():
            raise TypeError(f"Class name must start with uppercase: {name!r}")

        # Check method names
        for attr_name, attr_val in namespace.items():
            if callable(attr_val) and not attr_name.startswith('_'):
                if attr_name != attr_name.lower():
                    raise TypeError(
                        f"Public method must be snake_case: {attr_name!r}"
                    )

        return super().__new__(mcs, name, bases, namespace)


class GoodClass(metaclass=StrictNaming):
    def good_method(self):
        return "ok"


try:
    class badClass(metaclass=StrictNaming):   # lowercase start
        pass
except TypeError as e:
    print(f"TypeError: {e}")

try:
    class GoodButBadMethods(metaclass=StrictNaming):
        def badMethod(self):  # camelCase
            pass
except TypeError as e:
    print(f"TypeError: {e}")

print("GoodClass defined:", GoodClass().good_method())

---

## 15. Protocols (Structural Subtyping)

A **Protocol** (PEP 544, Python 3.8+) defines an interface by what methods an object has, not by what class it inherits from — known as "duck typing" made explicit for type checkers.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Drawable(Protocol):
    """Anything that can draw itself."""

    def draw(self) -> str:
        ...

    def bounding_box(self) -> tuple[float, float, float, float]:
        """Return (x_min, y_min, x_max, y_max)."""
        ...


# These classes don't inherit from Drawable — they just implement the methods
class Square:
    def __init__(self, x, y, size):
        self.x = x; self.y = y; self.size = size

    def draw(self) -> str:
        return f"□ Square at ({self.x},{self.y}) size={self.size}"

    def bounding_box(self):
        return (self.x, self.y, self.x+self.size, self.y+self.size)


class Triangle:
    def __init__(self, x, y, base, height):
        self.x = x; self.y = y; self.base = base; self.height = height

    def draw(self) -> str:
        return f"△ Triangle at ({self.x},{self.y})"

    def bounding_box(self):
        return (self.x, self.y, self.x+self.base, self.y+self.height)


# A function that accepts any Drawable — no inheritance required
def render_all(shapes: list[Drawable]) -> None:
    for shape in shapes:
        print(f"  {shape.draw()}  bbox={shape.bounding_box()}")


shapes = [Square(0, 0, 5), Triangle(2, 2, 4, 3)]
render_all(shapes)

print()
# runtime_checkable lets isinstance() work
print(f"Square is Drawable:   {isinstance(Square(0,0,1), Drawable)}")
print(f"str is Drawable:      {isinstance('hello', Drawable)}")

---

## 16. Design Patterns in Python

In [ ]:
# ── Singleton — only one instance ever created ────────────────────────────

class Singleton:
    """Only one instance of this class can exist."""
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value: int):
        self.value = value


a = Singleton(1)
b = Singleton(2)
print(f"Same instance: {a is b}")   # True
print(f"a.value={a.value}, b.value={b.value}")  # both see value=2

In [ ]:
# ── Observer / Event — notify many objects when something changes ──────────

from typing import Callable

class EventEmitter:
    """Pub-sub event system."""

    def __init__(self):
        self._listeners: dict[str, list[Callable]] = {}

    def on(self, event: str, callback: Callable) -> None:
        self._listeners.setdefault(event, []).append(callback)

    def off(self, event: str, callback: Callable) -> None:
        if event in self._listeners:
            self._listeners[event].remove(callback)

    def emit(self, event: str, *args, **kwargs) -> None:
        for cb in self._listeners.get(event, []):
            cb(*args, **kwargs)


class Store(EventEmitter):
    """A data store that emits events on change."""

    def __init__(self):
        super().__init__()
        self._data = {}

    def set(self, key: str, value) -> None:
        old = self._data.get(key)
        self._data[key] = value
        self.emit('change', key=key, old=old, new=value)


store = Store()
store.on('change', lambda key, old, new: print(f"  [{key}] {old!r} → {new!r}"))
store.on('change', lambda key, old, new: print(f"  (logged change to {key!r})"))

print("Making changes:")
store.set('user', 'Alice')
store.set('score', 100)
store.set('user', 'Bob')

In [ ]:
# ── Factory — create objects without specifying the exact class ───────────

class Serialiser:
    def serialise(self, data: dict) -> str:
        raise NotImplementedError

class JsonSerialiser(Serialiser):
    def serialise(self, data: dict) -> str:
        return json.dumps(data)

class CsvSerialiser(Serialiser):
    def serialise(self, data: dict) -> str:
        keys = list(data.keys())
        vals = [str(v) for v in data.values()]
        return ','.join(keys) + '\n' + ','.join(vals)

class XmlSerialiser(Serialiser):
    def serialise(self, data: dict) -> str:
        tags = ''.join(f"<{k}>{v}</{k}>" for k, v in data.items())
        return f"<record>{tags}</record>"


def get_serialiser(fmt: str) -> Serialiser:
    """Factory function — returns the right serialiser for fmt."""
    formats = {
        'json': JsonSerialiser,
        'csv':  CsvSerialiser,
        'xml':  XmlSerialiser,
    }
    if fmt not in formats:
        raise ValueError(f"Unknown format {fmt!r}. Choose from {list(formats)}")
    return formats[fmt]()


data = {'name': 'Alice', 'age': 30, 'city': 'NYC'}

for fmt in ['json', 'csv', 'xml']:
    s = get_serialiser(fmt)
    print(f"\n{fmt.upper()}:")
    print(s.serialise(data))

In [ ]:
# ── Strategy — swap algorithms at runtime ─────────────────────────────────

from typing import Protocol

class SortStrategy(Protocol):
    def sort(self, data: list) -> list:
        ...

class BubbleSort:
    def sort(self, data: list) -> list:
        data = list(data)
        for i in range(len(data)):
            for j in range(len(data) - i - 1):
                if data[j] > data[j + 1]:
                    data[j], data[j+1] = data[j+1], data[j]
        return data

class QuickSort:
    def sort(self, data: list) -> list:
        if len(data) <= 1:
            return data
        pivot = data[len(data) // 2]
        left  = [x for x in data if x < pivot]
        mid   = [x for x in data if x == pivot]
        right = [x for x in data if x > pivot]
        return self.sort(left) + mid + self.sort(right)

class Sorter:
    """Context — uses a sort strategy that can be swapped at runtime."""

    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy

    @property
    def strategy(self):
        return type(self._strategy).__name__

    def set_strategy(self, strategy: SortStrategy) -> None:
        self._strategy = strategy

    def sort(self, data: list) -> list:
        return self._strategy.sort(data)


data = [5, 1, 9, 3, 7, 2, 8, 4, 6]
sorter = Sorter(BubbleSort())
print(f"BubbleSort: {sorter.sort(data)}")

sorter.set_strategy(QuickSort())
print(f"QuickSort:  {sorter.sort(data)}")

---

## 17. Quick Reference

In [ ]:
quick_ref = """
OOP QUICK REFERENCE
====================

CORE CONCEPTS
  class Foo:              → define a class
  def __init__(self):     → initialiser
  self.attr = val         → instance attribute
  Foo.attr = val          → class attribute
  obj = Foo()             → create instance
  isinstance(obj, Foo)    → check type
  issubclass(Bar, Foo)    → check hierarchy

METHODS
  def method(self):       → instance method
  @classmethod            → receives cls
  def cls_method(cls):    → alternative constructors
  @staticmethod           → no self or cls
  def static_method():    → utility on the class

PROPERTIES
  @property               → getter
  @attr.setter            → setter with validation
  @attr.deleter           → deleter

INHERITANCE
  class Bar(Foo):         → inherit from Foo
  super().__init__(...)   → call parent __init__
  super().method(...)     → call parent method
  Foo.__mro__             → method resolution order

ABSTRACT
  class Foo(ABC):         → abstract base class
  @abstractmethod         → must be implemented by subclasses

DATACLASSES
  @dataclass              → auto-generate __init__, __repr__, __eq__
  @dataclass(frozen=True) → immutable + hashable
  @dataclass(order=True)  → auto comparison methods
  field(default_factory=) → mutable field defaults
  __post_init__           → post-init validation/computation
  asdict(obj)             → convert to dict
  replace(obj, attr=new)  → copy with changes

SLOTS
  __slots__ = ('x', 'y')  → fixed attribute set, lower memory

KEY DUNDERS
  __repr__ / __str__      → string representation
  __eq__ / __hash__       → equality and hashing
  __lt__ etc.             → comparison operators
  __add__ / __radd__      → arithmetic operators
  __len__ / __getitem__   → container protocol
  __iter__ / __next__     → iterator protocol
  __enter__ / __exit__    → context manager protocol
  __call__                → make instance callable
  __bool__                → truthiness

PROTOCOLS (typing.Protocol)
  Structural subtyping — define interface by methods, not base class
  @runtime_checkable      → allows isinstance() checks
"""
print(quick_ref)

In [ ]:
# 🏆 Practice: build a fully-featured Card and Deck system

from dataclasses import dataclass
from enum import Enum
import random


class Suit(Enum):
    CLUBS    = "♣"
    DIAMONDS = "♦"
    HEARTS   = "♥"
    SPADES   = "♠"


class Rank(Enum):
    TWO   = 2;  THREE = 3;  FOUR  = 4;  FIVE  = 5
    SIX   = 6;  SEVEN = 7;  EIGHT = 8;  NINE  = 9
    TEN   = 10; JACK  = 11; QUEEN = 12; KING  = 13; ACE = 14


@dataclass(frozen=True, order=True)
class Card:
    """
    A single playing card.

    frozen=True makes it immutable and hashable.
    order=True compares by (rank.value, suit.value).
    """
    rank: Rank
    suit: Suit

    def __str__(self) -> str:
        names = {11: 'J', 12: 'Q', 13: 'K', 14: 'A'}
        rank_str = names.get(self.rank.value, str(self.rank.value))
        return f"{rank_str}{self.suit.value}"


class Deck:
    """A standard 52-card deck."""

    def __init__(self):
        self._cards = [
            Card(rank, suit)
            for suit in Suit
            for rank in Rank
        ]

    def __len__(self) -> int:
        return len(self._cards)

    def __repr__(self) -> str:
        return f"Deck({len(self)} cards)"

    def __iter__(self):
        return iter(self._cards)

    def __getitem__(self, index):
        return self._cards[index]

    def shuffle(self) -> None:
        random.shuffle(self._cards)

    def deal(self, n: int = 1) -> list[Card]:
        if n > len(self._cards):
            raise ValueError(f"Cannot deal {n} cards from {len(self)} remaining")
        hand, self._cards = self._cards[:n], self._cards[n:]
        return hand

    def sort(self) -> None:
        self._cards.sort()


# Demo
deck = Deck()
print(f"New deck: {deck}")

deck.shuffle()
hand = deck.deal(5)
print(f"Hand dealt: {' '.join(str(c) for c in hand)}")
print(f"Remaining: {deck}")

# Cards are hashable (frozen dataclass) → usable in sets
hand_set = set(hand)
print(f"Unique cards in hand: {len(hand_set)}")

# Cards are comparable (order=True)
print(f"Highest card: {max(hand)}")
print(f"Sorted hand:  {' '.join(str(c) for c in sorted(hand))}")